# UltraLight VM-UNet — train & test on RTX 5060 (Blackwell)

Reproduces Table 1 of Wu et al., *Patterns* 6, 101298 (2025).

**Target (ISIC2017):** DSC 0.9091 · IoU 0.8334 · ACC 0.9646 · SE 0.9053 · SP 0.9790 ·
Prec 0.9481, at 0.049 M params / 0.060 GFLOPs.

The selective scan runs `mamba_ssm`'s fused CUDA kernel — the same code path the paper's
numbers come from. `models/mamba_pytorch.py` holds an independent pure-PyTorch
implementation of the same block, used only as the oracle the tests in cell 4 check that
kernel against.

### Setup (run once, in a terminal, before starting this notebook)

```bash
python3.13 -m venv .venv
source .venv/bin/activate
pip install torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu129
python scripts/install_mamba.py      # prebuilt mamba_ssm + causal_conv1d wheels
pip install -r requirements.txt
python -m ipykernel install --user --name ultralight --display-name "UltraLight"
```

Then pick the **UltraLight** kernel for this notebook.

Two version constraints, both about Blackwell (sm_120):

| | why |
|---|---|
| torch ≥ 2.7, CUDA ≥ 12.8 | 2.7.0 was the first stable release shipping sm_120 kernels. A cu117 build has kernels only to sm_86 and embeds just `compute_37` PTX — it cannot run here at all. |
| `mamba_ssm` ≥ 2.x, cu12/cu13 wheel | its setup.py emits an sm_120 cubin only when built against CUDA ≥ 12.8, and it ships no forward-compatible PTX. The `mamba_ssm==1.0.1` the paper pins predates that entirely. |

Cell 1 checks both by executing real kernels rather than by reading version strings.

## 1. Preflight — can this stack actually run on this GPU?

This is the cell that matters on new hardware. It does not trust version strings; it
executes a real torch kernel and a real fused selective scan, forward *and* backward, at
the smallest shape the model uses.

In [ ]:
import os, sys, platform
import torch

print('python  :', platform.python_version())
print('torch   :', torch.__version__)
print('cuda    :', torch.version.cuda)
print()

assert torch.cuda.is_available(), 'no CUDA device visible to torch'
name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
archs = torch.cuda.get_arch_list()
print(f'device  : {name}')
print(f'compute : sm_{cc[0]}{cc[1]}')
print(f'VRAM    : {vram:.1f} GB')
print(f'built   : {archs}')
print()

# The definitive check: run a real kernel rather than compare version strings.
try:
    a = torch.randn(512, 512, device='cuda')
    (a @ a).sum().item()
    torch.cuda.synchronize()
    print(f'OK - torch {torch.__version__} executes on sm_{cc[0]}{cc[1]}')
except RuntimeError as e:
    raise SystemExit(
        f'torch {torch.__version__} CANNOT execute on sm_{cc[0]}{cc[1]}.\n'
        f'It was built for {archs}.\n\n'
        'Blackwell needs torch >= 2.7 with CUDA >= 12.8:\n'
        '  pip install torch==2.10.0 torchvision==0.25.0 '
        '--index-url https://download.pytorch.org/whl/cu129\n\n'
        f'{type(e).__name__}: {e}')

# Same again for the CUDA extensions, which are compiled separately from torch and
# carry their own set of architectures. d_model=6 is the encoder4 branch width, the
# smallest the model uses; backward is a different kernel, so exercise it too.
import mamba_ssm
from mamba_ssm import Mamba
from mamba_ssm.modules import mamba_simple

print()
print('mamba_ssm      :', mamba_ssm.__version__)
print('causal_conv1d  :', 'present (fused mamba_inner_fn path)'
      if mamba_simple.causal_conv1d_fn is not None else 'MISSING (slower path)')
try:
    Mamba(d_model=6, d_state=16, d_conv=4, expand=2).cuda()(
        torch.randn(1, 64, 6, device='cuda')).sum().backward()
    torch.cuda.synchronize()
    print(f'OK - the fused selective scan executes on sm_{cc[0]}{cc[1]}')
except RuntimeError as e:
    raise SystemExit(
        f'mamba_ssm {mamba_ssm.__version__} CANNOT execute on sm_{cc[0]}{cc[1]}.\n'
        'The wheel has no cubin for this architecture and mamba_ssm ships no PTX.\n'
        'Install one built against CUDA >= 12.8:\n'
        '  python scripts/install_mamba.py\n\n'
        f'{type(e).__name__}: {e}')

## 2. Working directory

Everything below assumes the repository root, so the notebook behaves the same whether
the kernel starts in `notebooks/` or elsewhere.

In [ ]:
import os, sys

here = os.getcwd()
if os.path.basename(here) == 'notebooks':
    os.chdir('..')
ROOT = os.getcwd()
sys.path.insert(0, ROOT)

assert os.path.isfile('train.py'), f'expected the repository root, got {ROOT}'
print('working dir:', ROOT)
print('contents   :', sorted(d for d in os.listdir() if not d.startswith('.')))

## 3. Data

Pulls the six prepared `.npy` splits (524 MB) from HuggingFace. Preprocessing and the
train/val/test split were done **once** with `SPLIT_SEED = 42`, so every machine
consumes identical bytes and the split cannot drift.

The dataset repo is public, so this needs no token. (If you make it private, set
`HF_TOKEN` or run `huggingface-cli login` first.)

> The split matters. An earlier run used a *sorted* file listing, which looked
> reproducible but was biased: ISIC IDs correlate with acquisition source, giving
> train/val/test mean lesion areas of 22.9% / 8.0% / 15.0%. That alone cost 4.1 DSC
> points. The seeded permutation gives 20.0% / 17.6% / 18.6%.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/hf_data.py", "pull",
                "--dataset", "ISIC2017", "--repo", "RohanRamesh/ultralight-vmunet-data"], check=True)

In [ ]:
import numpy as np

print(f'{"split":6s} {"images":>7s} {"shape":>22s} {"lesion area":>12s}')
print('-' * 52)
for s in ('train', 'val', 'test'):
    d = np.load(f'data/ISIC2017/data_{s}.npy', mmap_mode='r')
    m = np.load(f'data/ISIC2017/mask_{s}.npy', mmap_mode='r')
    fg = (np.asarray(m) >= 128).mean() * 100
    print(f'{s:6s} {len(d):7d} {str(d.shape):>22s} {fg:11.1f}%')

# balanced splits are the fix described above; wildly different values mean stale data
print()
print('expect roughly 20.0 / 17.6 / 18.6 % -- if not, the data is from the old sorted split')

## 4. Sanity checks — do not skip

Seconds to run, and they catch the failures that would otherwise surface hours in.

The scan that runs here is `mamba_ssm`'s fused CUDA kernel, which is opaque from the
outside — so the tests pin it against `models/mamba_pytorch.py`, an independent
transcription of the reference scan from the official Mamba repository. They assert the
two agree at every one of the six real layer shapes, on the initial weights (bit for
bit), the forward pass, and every gradient, on *this* torch build and *this* GPU.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import copy
import torch
from models.UltraLight_VM_UNet import UltraLight_VM_UNet, MAMBA_BACKEND
from thop import profile

m = UltraLight_VM_UNet().cuda()
total = sum(p.numel() for p in m.parameters())
# profile a copy: thop leaves total_ops/total_params buffers on every submodule
flops, _ = profile(copy.deepcopy(m), inputs=(torch.randn(1, 3, 256, 256).cuda(),), verbose=False)

print('backend :', MAMBA_BACKEND)
print(f'params  : {total}   (paper: 49457)')
print(f'GFLOPs  : {flops/1e9:.4f}  (paper: 0.060)')
assert total == 49457, f'parameter count drifted: {total} != 49457'
print()
print('OK - structural match to the paper')
del m; torch.cuda.empty_cache()

## 5. (optional) Where does throughput peak on this GPU

The model is 0.049 M parameters, so **VRAM is not the limiting resource** — a batch of 8
uses well under 1 GB. The cost is per-launch overhead: the network is a long chain of
individually tiny operations, and the number of launches per forward is set by the
architecture, not by the batch size, so a larger batch amortises a fixed cost over more
images.

Useful for planning your own experiments. **Do not raise `config.batch_size` for the
replication run** — 8 is the paper's hyperparameter, and 32 would mean 39 optimiser steps
per epoch instead of 157, which changes the training trajectory and the result.

In [ ]:
!python scripts/bench_batch.py

## 6. Train

250 epochs, then automatic evaluation of the best checkpoint on the test split.

Measured on the RTX 5060: **21.7 min for 250 epochs** (5.2 s/epoch), plus ~1 min for the
600-image test pass and its overlay PNGs. For reference, run 1 took 1.64 h on a Kaggle T4
(~23 s/epoch) before the migration to `mamba_ssm`.

Only ~2 s of each epoch is GPU work; the rest is the CPU-side `scipy.ndimage.rotate`
augmentation on the main thread (`num_workers = 0`, deliberately — see the notes at the
bottom).

`checkpoints/latest.pth` is written every epoch and resumed automatically, so an
interrupted run continues where it stopped.

In [ ]:
!python train.py

## 7. Test a checkpoint on its own

`train.py` already evaluates the best checkpoint at the end. This is for re-evaluating
later without retraining — after a crash, or to compare checkpoints.

In [ ]:
import glob, os

runs = sorted(glob.glob('results/UltraLight_VM_UNet_ISIC2017_*'), key=os.path.getmtime)
assert runs, 'no run directories found -- has training been run?'
latest = runs[-1]
ckpts = sorted(glob.glob(os.path.join(latest, 'checkpoints', 'best-epoch*.pth')))
print('latest run :', latest)
print('checkpoints:', [os.path.basename(c) for c in ckpts] or '(none yet)')

In [ ]:
# evaluates the best checkpoint from the run above
import glob, os, subprocess, sys

ckpts = sorted(glob.glob(os.path.join(latest, 'checkpoints', 'best-epoch*.pth')))
if ckpts:
    subprocess.run([sys.executable, 'test.py', '--weights', ckpts[-1],
                    '--work-dir', latest], check=True)
else:
    print('no best-epoch checkpoint yet; run cell 6 first')

## 8. Results vs. the paper

In [ ]:
import ast, glob, re, os

PAPER = {'DSC': 0.9091, 'IoU': 0.8334, 'ACC': 0.9646,
         'SE': 0.9053, 'SP': 0.9790, 'Prec': 0.9481}

logs = sorted(glob.glob(os.path.join(latest, 'log', '*.log')), key=os.path.getmtime)
text = ''.join(open(p, encoding='utf-8', errors='replace').read() for p in logs)
hits = re.findall(r'test of best model.*?confusion_matrix: \[\[.*?\]\]', text, re.S)
assert hits, 'no test result in the log yet'
line = hits[-1]

def grab(k):
    m = re.search(k + r':\s*([\d.]+)', line)
    return float(m.group(1)) if m else float('nan')

# engine.py logs the raw confusion matrix but not precision, which the paper reports.
# Recover it: Prec = TP / (TP + FP).
cm = re.search(r'confusion_matrix: (\[\[.*?\]\])', line, re.S).group(1)
(TN, FP), (FN, TP) = ast.literal_eval(re.sub(r'\s+', ',', cm).replace('[,', '['))
print(f'confusion: TN {TN:,}  FP {FP:,}  FN {FN:,}  TP {TP:,}')
print(f'test loss: {grab("loss"):.4f}\n')

ours = {'DSC': grab('f1_or_dsc'), 'IoU': grab('miou'), 'ACC': grab('accuracy'),
        'SE': grab('sensitivity'), 'SP': grab('specificity'), 'Prec': TP / (TP + FP)}

# DSC is the primary metric and the one the +/-0.01 tolerance is set on. IoU is NOT
# independent: for a single foreground class IoU = DSC / (2 - DSC), so an IoU delta is
# just the DSC delta viewed through a steeper curve (~1.7x larger in this region). It is
# shown for completeness but must not be judged against the same threshold as DSC.
print(f'{"metric":8s} {"paper":>8s} {"ours":>8s} {"delta":>9s}')
print('-' * 40)
dsc_delta = ours['DSC'] - PAPER['DSC']
for k in ('DSC', 'IoU', 'SE', 'SP', 'ACC', 'Prec'):
    d = ours[k] - PAPER[k]
    if k == 'DSC':
        note = '  <-- primary; |d|<=0.01 = replicated' if abs(d) <= 0.01 else '  <-- primary; OUTSIDE +/-0.01'
    elif k == 'IoU':
        note = '  (derived from DSC, not independent)'
    else:
        note = ''
    print(f'{k:8s} {PAPER[k]:8.4f} {ours[k]:8.4f} {d:+9.4f}{note}')

print()
if abs(dsc_delta) <= 0.01:
    print(f'REPLICATED: DSC within {abs(dsc_delta)*100:.2f}% of the paper.')
else:
    print(f'DSC gap {dsc_delta:+.4f} is outside +/-0.01 -- worth investigating.')
print()
print('The paper gives no seed for its "random" split, so a different partition of the')
print('same 2000 images lands slightly differently. The cross-version torch/cuDNN stack')
print('adds a little more. A sub-1% DSC gap is a successful replication, not a defect.')

## 9. Plots


In [ ]:
from scripts.plot_metrics import generate_all
from IPython.display import Image, display

paths = generate_all(latest)
for p in paths:
    print(p)
    display(Image(filename=p))


---

## Notes

**Comparing across machines.** Results may differ slightly between machines even with an identical
split and seed — cuDNN convolution-algorithm selection is not guaranteed stable across torch
versions or GPU architectures, and neither is a fused scan across CUDA versions. Fine for a
standalone replication; worth a footnote if you put numbers from two machines in one table.

**8 GB is plenty.** Peak usage at batch 8 is around 1 GB. If you want to use the card harder, that
is what cell 5 is for — but see the warning there about `batch_size` and the paper.

**`num_workers = 0`** deliberately. Raising it hides the CPU-side `scipy.ndimage.rotate`
augmentation and is now by far the biggest remaining speedup — only ~2 s of each 5.2 s epoch is GPU
work. But each worker seeds its own RNG, so the augmentation stream — and therefore the training
trajectory — changes. Left alone so the replication stays comparable. Reasonable to enable once you
are benchmarking your own variants against your own baseline.

**Result of record (run 3):** DSC 0.9026 vs the paper's 0.9091 — a 0.71% gap, a successful
replication, and within 0.0004 DSC of run 2 on the pure-PyTorch scan.

**On the precision row.** The paper's Prec 0.9481 is not consistent with its own DSC and SE: DSC is
the harmonic mean of the two, and 0.9481/0.9053 give 0.9262, not the 0.9091 reported. The precision
implied by the paper's own DSC and SE is 0.9129, against our 0.9138. Do not read the −0.03 in the
table above as a model deficiency. Full working in `results/COMPARISON.md`.

**Full deviation list** from the reference implementation is in `README.md`.